In [1]:
import numpy as np
import pandas as pd

In [3]:
from urllib.parse import quote_plus

In [4]:
from sqlalchemy import create_engine

Establishment of connect from python notebook to MySQL

In [ ]:
username = 'root'
password = quote_plus('PASSWORD_HERE')
host = 'localhost'
port = 3306
database = 'project_inventory'

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

File paths


In [6]:
customers_dataset = r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\RAW DATA\olist_customers_dataset.csv"
geolocation_dataset = r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\RAW DATA\olist_geolocation_dataset.csv"
order_items_dataset = r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\RAW DATA\olist_order_items_dataset.csv"
order_payments_dataset = r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\RAW DATA\olist_order_payments_dataset.csv"
order_reviews_dataset = r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\RAW DATA\olist_order_reviews_dataset.csv"
orders_dataset = r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\RAW DATA\olist_orders_dataset.csv"
products_dataset = r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\RAW DATA\olist_products_dataset.csv"
sellers_dataset = r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\RAW DATA\olist_sellers_dataset.csv"
product_category = r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\RAW DATA\product_category_name_translation.csv"

Function used to replace strings in the column

In [7]:
def replace_str(column,from_word,to_word):
    column = column.str.replace(from_word,to_word)
    return column

Function used to extract date or time

In [8]:
def date_or_time_extractor(column,split_value,index_value):
    return column.str.split(split_value).str[index_value]
    

Function replaces the existing table in the DB

In [9]:
def replace_existing_table(tablename,engine,data_file):
    data_file.to_sql(name = tablename, con = engine,if_exists = 'replace',chunksize=10000, method='multi' )
    

In [103]:
cds = pd.read_csv(customers_dataset)

In [104]:
cds.shape

(99441, 5)

In [105]:
cds

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [106]:
cds.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


In [107]:
cds.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [108]:
cds['customer_id'].unique()

<StringArray>
['06b8999e2fba1a1fbc88172c00ba8bc7', '18955e83d337fd6b2def6b18a428ac77',
 '4e7b3e00288586ebd08712fdd0374a03', 'b2b6027bc5c5109e529d4dc6358b12c3',
 '4f2d8ab171c80ec8364f7c12e35b23ad', '879864dab9bc3047522c92c82e1212b8',
 'fd826e7cf63160e536e0908c76c3f441', '5e274e7a0c3809e14aba7ad5aae0d407',
 '5adf08e34b2e993982a47070956c5c65', '4b7139f34592b3a31687243a302fa75b',
 ...
 'be842c57a8c5a62e9585dd72f22b6338', 'f255d679c7c86c24ef4861320d5b7675',
 '14308d2303a3e2bdf4939b86c46d2679', 'f5a0b560f9e9427792a88bec97710212',
 '7fe2e80252a9ea476f950ae8f85b0f8f', '17ddf5dd5d51696bb3d7c6291687be6f',
 'e7b71a9017aa05c9a7fd292d714858e8', '5e28dfe12db7fb50a4b2f691faecea5e',
 '56b18e2166679b8a959d72dd06da27f9', '274fa6071e5e17fe303b9748641082c8']
Length: 99441, dtype: str

In [109]:
cds['customer_unique_id'].unique()

<StringArray>
['861eff4711a542e4b93843c6dd7febb0', '290c77bc529b7ac935b93aa66c333dc3',
 '060e732b5b29e8181a18229c7b0b2b5e', '259dac757896d24d7702b9acbbff3f3c',
 '345ecd01c38d18a9036ed96c73b8d066', '4c93744516667ad3b8f1fb645a3116a4',
 'addec96d2e059c80c30fe6871d30d177', '57b2a98a409812fe9618067b6b8ebe4f',
 '1175e95fb47ddff9de6b2b06188f7e0d', '9afe194fb833f79e300e37e580171f22',
 ...
 'ca186065de6e2d01cfc99763e6d62048', 'd111b06b6f3a2add0d2241325f65b5ca',
 'e7f8760e2bbd2f1986bebd99596c088e', 'b3e53d18a997f27a3ffd16da497eaf58',
 '4b5820135d360a45552b5163835b1d89', '1a29b476fee25c95fbafc67c5ac95cf8',
 'd52a67c98be1cf6a5c84435bd38d095d', 'e9f50caf99f032f0bf3c55141f019d99',
 '73c2643a0a458b49f58cea58833b192e', '84732c5050c01db9b23e19ba39899398']
Length: 96096, dtype: str

In [110]:
cds['customer_state'].unique()

<StringArray>
['SP', 'SC', 'MG', 'PR', 'RJ', 'RS', 'PA', 'GO', 'ES', 'BA', 'MA', 'MS', 'CE',
 'DF', 'RN', 'PE', 'MT', 'AM', 'AP', 'AL', 'RO', 'PB', 'TO', 'PI', 'AC', 'SE',
 'RR']
Length: 27, dtype: str

In [111]:
cds['customer_zip_code_prefix'].unique()

array([14409,  9790,  1151, ...,  5538, 74980, 99043], shape=(14994,))

In [112]:
gds = pd.read_csv(geolocation_dataset)

In [113]:
gds


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


In [178]:
gds.value_counts()

geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  geolocation_city  geolocation_state
88220                        -27.102099       -48.629613       itapema           SC                   314
6414                         -23.495901       -46.874687       barueri           SP                   189
5145                         -23.506049       -46.717377       sao paulo         SP                   141
6414                         -23.490618       -46.869004       barueri           SP                   127
22620                        -23.005514       -43.375964       rio de janeiro    RJ                   102
                                                                                                     ... 
99965                        -28.180655       -52.034367       agua santa        RS                     1
99950                        -28.072188       -52.011272       tapejara          RS                     1
                             -28.068864       -52.012

In [115]:
gds.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), int64(1), str(2)
memory usage: 38.2 MB


In [116]:
gds.isnull().sum()

geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [179]:
gds.to_csv(
    r'D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\CLEANED DATA\cleaned_geolocation_data.csv',
    index=False
)

In [117]:
gds

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP
...,...,...,...,...,...
1000158,99950,-28.068639,-52.010705,tapejara,RS
1000159,99900,-27.877125,-52.224882,getulio vargas,RS
1000160,99950,-28.071855,-52.014716,tapejara,RS
1000161,99980,-28.388932,-51.846871,david canabarro,RS


In [118]:
gds['geolocation_city'].unique()

<StringArray>
[            'sao paulo',             'são paulo', 'sao bernardo do campo',
               'jundiaí',       'taboão da serra',              'sãopaulo',
                    'sp',            'sa£o paulo',   'sao jose dos campos',
                'osasco',
 ...
       'ipiranga do sul',          'vila langaro',               'ciriaco',
      'floriano peixoto',              'erebango',                'ibiaçá',
  'santa cecilia do sul',               'ciríaco',               'estação',
          'vila lângaro']
Length: 8011, dtype: str

In [119]:
gds['geolocation_city'] = replace_str(gds['geolocation_city'],'são paulo','sao paulo')

In [120]:
gds['geolocation_city'].unique()

<StringArray>
[            'sao paulo', 'sao bernardo do campo',               'jundiaí',
       'taboão da serra',              'sãopaulo',                    'sp',
            'sa£o paulo',   'sao jose dos campos',                'osasco',
           'carapicuíba',
 ...
       'ipiranga do sul',          'vila langaro',               'ciriaco',
      'floriano peixoto',              'erebango',                'ibiaçá',
  'santa cecilia do sul',               'ciríaco',               'estação',
          'vila lângaro']
Length: 8008, dtype: str

In [121]:
gds['geolocation_city'] = replace_str(gds['geolocation_city'],'sãopaulo','sao paulo')

In [122]:
gds['geolocation_city'] = replace_str(gds['geolocation_city'],'sp','sao paulo')

In [123]:
gds['geolocation_city'] = replace_str(gds['geolocation_city'],'sa£o paulo','sao paulo')

In [124]:
gds['geolocation_state'].unique()

<StringArray>
['SP', 'RN', 'AC', 'RJ', 'ES', 'MG', 'BA', 'SE', 'PE', 'AL', 'PB', 'CE', 'PI',
 'MA', 'PA', 'AP', 'AM', 'RR', 'DF', 'GO', 'RO', 'TO', 'MT', 'MS', 'RS', 'PR',
 'SC']
Length: 27, dtype: str

In [125]:
gds.to_sql(name='olist_geolocation_dataset',con=engine,if_exists='replace',index=False,chunksize=10000,method='multi')

1000163

In [126]:
oids = pd.read_csv(order_items_dataset)

In [127]:
oids

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [128]:
oids['shipping_limit_time']=oids['shipping_limit_date'].str.split(' ').str[1]

In [129]:
oids['new_shipping_limit_date']=oids['shipping_limit_date'].str.split(' ').str[0]

In [130]:
oids.drop('shipping_limit_date',axis=1,inplace=True)

In [131]:
oids

,order_id,order_item_id,product_id,seller_id,price,freight_value,shipping_limit_time,new_shipping_limit_date
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.90,13.29,09:45:35,2017-09-19
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.90,19.93,11:05:13,2017-05-03
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.00,17.87,14:48:30,2018-01-18
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79,10:10:18,2018-08-15
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.90,18.14,13:57:51,2017-02-13
...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,299.99,43.41,04:11:01,2018-05-02
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,350.00,36.53,04:31:48,2018-07-20
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,99.90,16.95,17:14:25,2017-10-30
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,55.99,8.72,00:04:32,2017-08-21


In [132]:
oids['new_shipping_limit_date'] = pd.to_datetime(oids['new_shipping_limit_date']).dt.date


In [133]:
oids.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 8 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   order_id                 112650 non-null  str    
 1   order_item_id            112650 non-null  int64  
 2   product_id               112650 non-null  str    
 3   seller_id                112650 non-null  str    
 4   price                    112650 non-null  float64
 5   freight_value            112650 non-null  float64
 6   shipping_limit_time      112650 non-null  object 
 7   new_shipping_limit_date  112650 non-null  object 
dtypes: float64(2), int64(1), object(2), str(3)
memory usage: 6.9+ MB


In [134]:
oids['product_id'].unique()

<StringArray>
['4244733e06e7ecb4970a6e2683c13e61', 'e5f2d52b802189ee658865ca93d83a8f',
 'c777355d18b72b67abbeef9df44fd0fd', '7634da152a4610f1595efa32f14722fc',
 'ac6c3623068f30de03045865e4e10089', 'ef92defde845ab8450f9d70c526ef70f',
 '8d4f2bb7e93e6710a28f34fa83ee7d28', '557d850972a7d6f792fd18ae1400d9b6',
 '310ae3c140ff94b03219ad0adc3c778f', '4535b0e1091c278dfd193e5a1d63b39f',
 ...
 '16b4473e98422039c388f144a0b16f55', 'dd44ecaddb22d00c140856b180f5d9b4',
 'e97df839917a6e93404867b1d0319bfc', 'fda6a1e500285956c972ec2fe10f4923',
 '801a695ff5c0c14970a71a4ceb70989e', '4cc4d02efc8f249c13355147fb44e34d',
 'b10ecf8e33aaaea419a9fa860ea80fb5', 'dd469c03ad67e201bc2179ef077dcd48',
 'bbe7651fef80287a816ead73f065fc4b', '350688d9dc1e75ff97be326363655e01']
Length: 32951, dtype: str

In [135]:
tablename_for_oids = 'olist_order_items_dataset'

In [136]:
replace_existing_table(tablename_for_oids,engine,oids)

In [181]:
oids.to_csv(
    r'D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\CLEANED DATA\cleaned_order_items_data.csv',
    index=False
)

In [137]:
opds = pd.read_csv(order_payments_dataset)

In [138]:
opds

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [182]:
opds.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [140]:
opds['payment_installments'].unique()

array([ 8,  1,  2,  3,  6,  5,  4, 10,  7, 12,  9, 13, 15, 24, 11, 18, 14,
       20, 21, 17, 22,  0, 16, 23])

In [183]:
opds.shape

(103886, 5)

In [141]:
ords = pd.read_csv(order_reviews_dataset)

In [142]:
ords

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


In [143]:
ords.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [144]:
ords['review_creation_date'] = date_or_time_extractor(ords['review_creation_date'],' ',0)

In [145]:
ords

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01,2018-07-02 12:59:13


In [146]:
ords['review_answer_timestamp'] = date_or_time_extractor(ords['review_answer_timestamp'],' ',1)

In [147]:
ords

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07,17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09,20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22,09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01,12:59:13


In [32]:
table_name_for_ords = 'olist_order_reviews_dataset'

replace_existing_table(table_name_for_ords,engine,ords)

NameError: name 'ords' is not defined

In [10]:
orders_ds = pd.read_csv(orders_dataset)

In [11]:
orders_ds

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [12]:
orders_ds.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [13]:

orders_ds['estimated_delivery_date'] = date_or_time_extractor(orders_ds['order_estimated_delivery_date'],' ',0)
orders_ds['order_purchase_date'] = date_or_time_extractor(orders_ds['order_purchase_timestamp'],' ',0)
orders_ds['order_purchase_time'] = date_or_time_extractor(orders_ds['order_purchase_timestamp'],' ',1)
orders_ds['order_approved_date'] = date_or_time_extractor(orders_ds['order_approved_at'],' ',0)
orders_ds['order_approved_time'] = date_or_time_extractor(orders_ds['order_approved_at'],' ',1)
orders_ds['new_order_delivered_carrier_date'] = date_or_time_extractor(orders_ds['order_delivered_carrier_date'],' ',0)
orders_ds['order_delivered_carrier_time'] = date_or_time_extractor(orders_ds['order_delivered_carrier_date'],' ',1)
orders_ds['new_order_delivered_customer_date'] = date_or_time_extractor(orders_ds['order_delivered_customer_date'],' ',0)
orders_ds['_neworder_delivered_customer_time'] = date_or_time_extractor(orders_ds['order_delivered_customer_date'],' ',1)
orders_ds['order_estimated_delivery_date'] = date_or_time_extractor(orders_ds['order_estimated_delivery_date'],' ',0)

In [14]:
orders_ds.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'estimated_delivery_date', 'order_purchase_date', 'order_purchase_time',
       'order_approved_date', 'order_approved_time',
       'new_order_delivered_carrier_date', 'order_delivered_carrier_time',
       'new_order_delivered_customer_date',
       '_neworder_delivered_customer_time'],
      dtype='str')

In [15]:
tb_removed = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date']
orders_ds.drop(columns= tb_removed, inplace=True)

In [16]:

orders_ds.reset_index(drop=True,inplace=True)

In [17]:
orders_ds

,order_id,customer_id,order_status,estimated_delivery_date,order_purchase_date,order_purchase_time,order_approved_date,order_approved_time,new_order_delivered_carrier_date,order_delivered_carrier_time,new_order_delivered_customer_date,_neworder_delivered_customer_time
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-18,2017-10-02,10:56:33,2017-10-02,11:07:15,2017-10-04,19:55:00,2017-10-10,21:25:13
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-08-13,2018-07-24,20:41:37,2018-07-26,03:24:27,2018-07-26,14:31:00,2018-08-07,15:27:45
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-09-04,2018-08-08,08:38:49,2018-08-08,08:55:23,2018-08-08,13:50:00,2018-08-17,18:06:29
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-12-15,2017-11-18,19:28:06,2017-11-18,19:45:59,2017-11-22,13:39:59,2017-12-02,00:28:42
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-26,2018-02-13,21:18:39,2018-02-13,22:20:29,2018-02-14,19:46:34,2018-02-16,18:17:02
...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-28,2017-03-09,09:54:05,2017-03-09,09:54:05,2017-03-10,11:18:03,2017-03-17,15:08:01
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-03-02,2018-02-06,12:58:58,2018-02-06,13:10:37,2018-02-07,23:22:42,2018-02-28,17:37:56
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-09-27,2017-08-27,14:46:43,2017-08-27,15:04:16,2017-08-28,20:52:26,2017-09-21,11:24:17
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-02-15,2018-01-08,21:28:27,2018-01-08,21:36:21,2018-01-12,15:35:03,2018-01-25,23:32:54


In [20]:
tablename_for_orders_ds = 'cleaned_orders_data'

replace_existing_table(tablename_for_orders_ds,engine,orders_ds)

In [157]:
pds = pd.read_csv(products_dataset)

In [158]:
pds.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [159]:
pds

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [160]:
pds.groupby('product_category_name')['product_weight_g'].count()

product_category_name
agro_industria_e_comercio      74
alimentos                      82
alimentos_bebidas             104
artes                          55
artes_e_artesanato             19
                             ... 
sinalizacao_e_seguranca        93
tablets_impressao_imagem        9
telefonia                    1134
telefonia_fixa                116
utilidades_domesticas        2335
Name: product_weight_g, Length: 73, dtype: int64

In [161]:
selds = pd.read_csv(sellers_dataset)

In [162]:
selds.shape

(3095, 4)

In [163]:
selds

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [164]:
selds['seller_city'].unique()

<StringArray>
[              'campinas',             'mogi guacu',         'rio de janeiro',
              'sao paulo',      'braganca paulista',                 'brejao',
              'penapolis',               'curitiba',               'anapolis',
              'itirapina',
 ...
          'varzea alegre',          'guaratingueta',                 'tambau',
                  'irati',          'riberao preto',   'aparecida de goiania',
           'bandeirantes', 'vitoria de santo antao',               'palotina',
                   'leme']
Length: 611, dtype: str

In [165]:
selds.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [166]:
translated =  pd.read_csv(product_category)

In [167]:
new = np.array(translated)

In [168]:
type(new)

numpy.ndarray

In [169]:
new.shape

(71, 2)

In [170]:
pds

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [171]:
new = dict(new)

In [172]:
new

{'beleza_saude': 'health_beauty',
 'informatica_acessorios': 'computers_accessories',
 'automotivo': 'auto',
 'cama_mesa_banho': 'bed_bath_table',
 'moveis_decoracao': 'furniture_decor',
 'esporte_lazer': 'sports_leisure',
 'perfumaria': 'perfumery',
 'utilidades_domesticas': 'housewares',
 'telefonia': 'telephony',
 'relogios_presentes': 'watches_gifts',
 'alimentos_bebidas': 'food_drink',
 'bebes': 'baby',
 'papelaria': 'stationery',
 'tablets_impressao_imagem': 'tablets_printing_image',
 'brinquedos': 'toys',
 'telefonia_fixa': 'fixed_telephony',
 'ferramentas_jardim': 'garden_tools',
 'fashion_bolsas_e_acessorios': 'fashion_bags_accessories',
 'eletroportateis': 'small_appliances',
 'consoles_games': 'consoles_games',
 'audio': 'audio',
 'fashion_calcados': 'fashion_shoes',
 'cool_stuff': 'cool_stuff',
 'malas_acessorios': 'luggage_accessories',
 'climatizacao': 'air_conditioning',
 'construcao_ferramentas_construcao': 'construction_tools_construction',
 'moveis_cozinha_area_de_ser

In [173]:
pds['product_category_name'] = pds['product_category_name'].replace(new)

In [174]:
pds

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,art,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,sports_leisure,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,baby,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,housewares,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,furniture_decor,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construction_tools_lights,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,bed_bath_table,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,computers_accessories,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [175]:
table_name_for_pds = 'olist_products_dataset'

In [176]:
replace_existing_table(table_name_for_pds,engine,pds)

In [7]:
df=pd.read_csv(r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\CLEANED DATA\cleaned_products_data.csv")

In [11]:
df.drop(columns=['index'],inplace=True)

In [12]:
df.to_csv(r"D:\DATA ANALYSIS PROJECT\BRAZILIAN OLIST\CLEANED DATA\cleaned_products_data.csv",index=False)